# Descriptive Statistics for the Cleaned Accident Dataset

This notebook describes the cleaned Kenya road accident dataset using summary statistics, frequency tables, distribution diagnostics, and count-data checks.

Focus areas:
- dataset structure and data quality
- mean, median, standard deviation, quartiles, skewness, and kurtosis
- categorical composition of the data
- distribution of accident timing and fatal outcomes
- Poisson-style assessment for the `num_victims` count variable


In [ ]:
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11


In [ ]:
DATA_PATH = '../data/cleaned/accidents_clean.csv'
df = pd.read_csv(DATA_PATH)

# Create numeric helper columns for statistical description.
df['time_numeric'] = pd.to_numeric(df['time'], errors='coerce')
df['hour_numeric'] = pd.to_numeric(df['hour'], errors='coerce')
df['age_numeric'] = pd.to_numeric(df['age'], errors='coerce')
df['cause_code_numeric'] = pd.to_numeric(df['cause_code'], errors='coerce')
df['num_victims_numeric'] = pd.to_numeric(df['num_victims'], errors='coerce')
df['is_fatal_numeric'] = pd.to_numeric(df['is_fatal'], errors='coerce')
df['date'] = pd.to_datetime(df['date'], errors='coerce')

print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')
df.head()


In [ ]:
summary = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_values': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2),
    'unique_values': df.nunique(dropna=True)
}).sort_values(['missing_values', 'unique_values'], ascending=[False, False])

summary


## Numeric Descriptive Statistics

The table below reports classical descriptive statistics for the main numeric fields. In addition to mean and standard deviation, it includes median, quartiles, skewness, and kurtosis to show distribution shape.

In [ ]:
numeric_cols = [
    'time_numeric',
    'hour_numeric',
    'age_numeric',
    'cause_code_numeric',
    'num_victims_numeric',
    'is_fatal_numeric'
]

numeric_df = df[numeric_cols].copy()
desc = numeric_df.describe().T
desc['median'] = numeric_df.median()
desc['variance'] = numeric_df.var()
desc['skewness'] = numeric_df.skew()
desc['kurtosis'] = numeric_df.kurtosis()
desc = desc[['count', 'mean', 'std', 'variance', 'min', '25%', '50%', '75%', 'max', 'median', 'skewness', 'kurtosis']]
desc.round(3)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(df['age_numeric'].dropna(), bins=20, kde=True, ax=axes[0, 0], color='#2f6fc4')
axes[0, 0].set_title('Age Distribution')

sns.histplot(df['hour_numeric'].dropna(), bins=24, kde=False, ax=axes[0, 1], color='#b97a00')
axes[0, 1].set_title('Hour of Day Distribution')

sns.countplot(data=df, x='time_of_day', order=df['time_of_day'].value_counts().index, ax=axes[1, 0], color='#2c8b46')
axes[1, 0].set_title('Time of Day Frequency')
axes[1, 0].tick_params(axis='x', rotation=20)

sns.countplot(data=df, x='is_fatal', ax=axes[1, 1], color='#b14a4a')
axes[1, 1].set_title('Fatal vs Non-Fatal Counts')
axes[1, 1].set_xticklabels(['Non-Fatal', 'Fatal'])

plt.tight_layout()
plt.show()


## Categorical Description

These frequency tables show how records are distributed across counties, victim groups, time of day, gender, and age groups.

In [ ]:
def frequency_table(series, top_n=10):
    counts = series.value_counts(dropna=False).head(top_n)
    return pd.DataFrame({
        'count': counts,
        'percent': (counts / len(series) * 100).round(2)
    })

display(frequency_table(df['county'], 10))
display(frequency_table(df['victim_category'], 10))
display(frequency_table(df['time_of_day'], 10))
display(frequency_table(df['gender_clean'], 10))
display(frequency_table(df['age_group'], 10))


In [ ]:
county_fatality = (
    df.groupby('county', dropna=False)
      .agg(accidents=('county', 'size'), fatalities=('is_fatal_numeric', 'sum'))
      .assign(fatality_rate=lambda x: x['fatalities'] / x['accidents'])
      .sort_values('accidents', ascending=False)
)

county_fatality.head(10)


## Poisson Distribution Check for `num_victims`

`num_victims` is a count variable, so a Poisson comparison is a relevant descriptive step. A Poisson process typically has mean approximately equal to variance. If the variance is much larger than the mean, the data are over-dispersed relative to a Poisson distribution.

In [ ]:
victim_counts = df['num_victims_numeric'].dropna().astype(int)
lam = victim_counts.mean()
var = victim_counts.var()
dispersion_ratio = var / lam if lam != 0 else np.nan

observed = victim_counts.value_counts().sort_index()
k_values = observed.index.to_list()

def poisson_pmf(k, lam):
    return math.exp(-lam) * (lam ** k) / math.factorial(k)

expected = pd.Series({k: poisson_pmf(k, lam) * len(victim_counts) for k in k_values})

poisson_summary = pd.DataFrame({
    'observed_count': observed,
    'expected_poisson_count': expected.round(2)
})

print(f'Mean of num_victims: {lam:.3f}')
print(f'Variance of num_victims: {var:.3f}')
print(f'Variance-to-mean ratio: {dispersion_ratio:.3f}')
print('A ratio close to 1 supports Poisson-like behavior; a much larger ratio suggests over-dispersion.')

poisson_summary


In [ ]:
plt.figure(figsize=(10, 5))
x = np.array(k_values)
plt.bar(x - 0.15, observed.values, width=0.3, label='Observed', color='#2c8b46')
plt.bar(x + 0.15, expected.values, width=0.3, label='Expected (Poisson)', color='#b97a00')
plt.xlabel('Number of Victims')
plt.ylabel('Frequency')
plt.title('Observed vs Poisson-Expected Counts for num_victims')
plt.legend()
plt.show()


## Interpretation Notes

Use the outputs above to write up the data description in your report. A concise interpretation can cover:
- central tendency: what the mean and median suggest
- spread: whether the standard deviation and variance are large or small
- shape: whether skewness suggests asymmetry
- class balance: fatal vs non-fatal proportion
- spatial concentration: which counties dominate the records
- count-data behavior: whether `num_victims` behaves like a Poisson variable or shows over-dispersion
